<a href="https://colab.research.google.com/github/joseportocarrero-stack/DataScience-Homework/blob/main/FGD_C28R_1A_LABD10_Portocarrero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LABORATORIO DIRIGIDO N.° 10
## Migración de datos y ETL en Python: Caso Farmacia MediSur

**Curso:** Fundamentos de Gestión de Datos
**Semana:** 10 — Migración de Datos y ETL
**Tema:** Pipeline ETL con Python, Pandas y SQLite sobre la base oficial del proyecto
**Docente:** Pilar Rocío Sayán Mejía
**Duración estimada:** 1 hora y 40 minutos

---

> **Importante — esta base te acompaña todo el proyecto (semanas 10 a 16).**
> Aquí **no se inventan datos**. Trabajarás con la **base oficial de tu caso**, alojada en GitHub. Es la misma base que documentarás (S11), analizarás (S12), medirás (S13) y gobernarás (S14–15) hasta el proyecto final PMD2. Lo que limpies hoy es el punto de partida de todo lo que sigue.

## Caso introductorio

Eres analista de datos en **Farmacia MediSur**, una cadena de farmacias con varias sucursales. Cada sucursal registró sus ventas y clientes de manera distinta y el sistema antiguo acumuló errores. Tu tarea de esta semana **no** es construir todo el proyecto final, sino realizar una **primera migración controlada** de la base oficial:

1. **Descargar** la base oficial del caso desde el repositorio (GitHub).
2. **Extraer** los datos desde SQLite.
3. **Diagnosticar** problemas de calidad e integridad.
4. **Transformar** y limpiar los datos con funciones de Pandas.
5. **Cargar** los datos limpios en una nueva base SQLite.
6. **Unificar** una segunda fuente que llega con columnas distintas.
7. **Automatizar** la carga con una función reutilizable (stored procedure simulado).
8. **Validar** la migración completa con un checklist.

Este laboratorio trabaja solamente los temas de la **Semana 10: migración de datos y ETL**.

## Actividad 1: conceptos previos

Completa con tus propias palabras:

| Concepto | Respuesta |
|---|---|
| ¿Qué significa ETL? | Es un acrónimo de los procesos Extracción, Transformación y carga, por sus siglas en inglés.|
| ¿Qué ocurre en la fase Extract? | Se mueven o copian los datos desde sus orígenes.|
| ¿Qué ocurre en la fase Transform? | Se limpian y adaptan los datos al formato de destino, quedando generalmente como datos estructurados.|
| ¿Qué ocurre en la fase Load? | Se guardan los datos en la base de destino y se validan.|
| ¿Por qué se valida una migración? | Se valida una migración por la necesidad de tener datos completos e íntegros al final de los procesos.|

## Actividad 2: desarrollo práctico en Colab

### Paso 1: importar librerías
Cargamos las librerías del sílabo: `sqlite3` para las bases, `pandas` para transformar y `numpy` para apoyo numérico.

In [ ]:
# Paso 1: importar librerias
import sqlite3
import os

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
np.random.seed(10)

try:
    display
except NameError:
    display = print

print("Entorno listo para el Laboratorio D10 - ETL con Farmacia MediSur")

Entorno listo para el Laboratorio D10 - ETL con Farmacia MediSur


### Paso 2: descargar la base OFICIAL del caso desde GitHub
La base **ya existe**: es la base oficial de tu caso publicada en el repositorio. La descargamos una sola vez.

> Cambia `caso` por la carpeta de **tu** caso asignado. El nombre debe coincidir **exactamente** con la carpeta del repositorio.

In [ ]:
# Paso 2: descargar la base oficial del caso (una sola vez)
caso = "03_farmacia_medisur"   # <-- cambia por la carpeta de TU caso asignado
DB_ORIGEN = "farmacia_legacy.db"
DB_DESTINO = "farmacia_migrada.db"
url = f"https://raw.githubusercontent.com/Rociosayan/PMD2_FDG_Casos/main/casos/{caso}/{caso}.db"

if not os.path.exists(DB_ORIGEN):
    import requests
    r = requests.get(url)
    if r.status_code == 200 and r.content[:16] == b"SQLite format 3\x00":
        with open(DB_ORIGEN, "wb") as f:
            f.write(r.content)
        print("Base oficial descargada desde GitHub:", len(r.content), "bytes")
    else:
        from google.colab import files
        print("No se pudo descargar. Sube manualmente el .db de tu caso:")
        subida = files.upload()
        with open(DB_ORIGEN, "wb") as f:
            f.write(list(subida.values())[0])
else:
    print("La base ya estaba disponible en el entorno:", DB_ORIGEN)

if os.path.exists(DB_DESTINO):
    os.remove(DB_DESTINO)
print("Base origen :", DB_ORIGEN)
print("Base destino:", DB_DESTINO)

Base oficial descargada desde GitHub: 344064 bytes
Base origen : farmacia_legacy.db
Base destino: farmacia_migrada.db


### Paso 3: EXTRACT — explorar la base y extraer la tabla de clientes
La base oficial es **relacional**: tiene varias tablas conectadas (sedes, empleados, clientes, productos/servicios, operaciones, detalle, pagos, incidencias). Primero vemos qué hay y cuántos registros trae cada tabla; luego extraemos la tabla `clientes`, que será el centro de nuestra limpieza.

In [ ]:
# Paso 3: EXTRACT - explorar tablas y extraer clientes
conn_origen = sqlite3.connect(DB_ORIGEN)

tablas = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn_origen)
print("Tablas en la base oficial:")
for t in tablas["name"]:
    n = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {t}", conn_origen)["n"][0]
    print(f"   - {t:22s} {n:5d} registros")

df_clientes = pd.read_sql_query("SELECT * FROM clientes", conn_origen)
print("\nRegistros extraidos de clientes:", len(df_clientes))
print("Columnas:", list(df_clientes.columns))
display(df_clientes.head())

Tablas en la base oficial:
   - clientes                 208 registros
   - detalle_operacion       2572 registros
   - empleados                 40 registros
   - incidencias              155 registros
   - operaciones             1010 registros
   - pagos                   1000 registros
   - productos_servicios       30 registros
   - sedes                      6 registros

Registros extraidos de clientes: 208
Columnas: ['id_cliente', 'codigo_cliente', 'tipo_documento', 'num_documento', 'nombres', 'apellidos', 'correo', 'telefono', 'distrito', 'segmento', 'fecha_registro', 'condicion_cronica']


,id_cliente,codigo_cliente,tipo_documento,num_documento,nombres,apellidos,correo,telefono,distrito,segmento,fecha_registro,condicion_cronica
0,1,C0001,RUC,79941052,Iván,Aguirre Bautista,iván1@correo.com,923380330,Rímac,Corporativo,2026-01-24,Hipertensión
1,2,C0002,DNI,377593,Melissa,Torres Lévano,melissa2@correo.com,966971049,Villa El Salvador,Corporativo,2026-05-18,Asma
2,3,C0003,DNI,38119160,Milagros,Villanueva Zevallos,milagros3@correo.com,911285064,Surco,Preferente,2026-03-08,Ninguna
3,4,C0004,DNI,14282620,Karla,Ríos Ninaquispe,karla4@correo.com,999382396,Villa El Salvador,Regular,2026-05-10,Ninguna
4,5,C0005,DNI,12191834,Hugo,Romero Castillo,hugo5@correo.com,982000067,Surco,Corporativo,2026-04-20,Asma


**Pregunta 1:** ¿Cuántas tablas tiene la base y cuántos registros tiene `clientes`? ¿Por qué se dice que es una base *relacional*?

**Respuesta:** _La base tiene 8 tablas; la tabla clientes cuenta con 208 registros. Se dice que es una base de datos relacional porque almacena sus datos en filas y columnas, formando tablas relacionadas entre sí por claves o identificadores._

### Paso 4: diagnóstico de CALIDAD de la tabla clientes
Antes de limpiar, medimos el daño. Revisamos valores nulos, documentos (DNI) duplicados y correos mal escritos.

In [ ]:
# Paso 4: diagnostico de calidad de clientes
print("Nulos por columna:")
display(df_clientes.isna().sum())

dup_doc = int(df_clientes["num_documento"].duplicated().sum())
correo = df_clientes["correo"].dropna()
correos_malos = int((~correo.str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", na=False)).sum())

print("\nDocumentos (num_documento) duplicados:", dup_doc)
print("Correos con formato invalido:", correos_malos)
print("Filas completamente duplicadas:", int(df_clientes.duplicated().sum()))

Nulos por columna:


,0
id_cliente,0
codigo_cliente,0
tipo_documento,0
num_documento,0
nombres,0
apellidos,0
correo,14
telefono,18
distrito,10
segmento,0



Documentos (num_documento) duplicados: 6
Correos con formato invalido: 11
Filas completamente duplicadas: 0


**Pregunta 2:** ¿Qué problemas de calidad observas en `clientes` antes de transformar?

**Respuesta:** _Se observan registros con número de cliente duplicado, valores nulos y direcciones de correo con formato inválido._

### Paso 5: diagnóstico de INTEGRIDAD y montos
En una base relacional también fallan las **relaciones**: ventas que apuntan a un cliente o empleado que no existe (registros *huérfanos*) y montos imposibles (negativos). Los detectamos con `LEFT JOIN`.

In [ ]:
# Paso 5: diagnostico de integridad referencial y montos
op_sin_cliente = pd.read_sql_query("""
    SELECT COUNT(*) AS n FROM operaciones o
    LEFT JOIN clientes c ON o.id_cliente = c.id_cliente
    WHERE c.id_cliente IS NULL""", conn_origen)["n"][0]
op_sin_empleado = pd.read_sql_query("""
    SELECT COUNT(*) AS n FROM operaciones o
    LEFT JOIN empleados e ON o.id_empleado = e.id_empleado
    WHERE e.id_empleado IS NULL""", conn_origen)["n"][0]
montos_negativos = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM operaciones WHERE monto_total < 0", conn_origen)["n"][0]

print("Ventas huerfanas (sin cliente valido):", op_sin_cliente)
print("Ventas huerfanas (sin empleado valido):", op_sin_empleado)
print("Ventas con monto_total negativo:", montos_negativos)

Ventas huerfanas (sin cliente valido): 15
Ventas huerfanas (sin empleado valido): 22
Ventas con monto_total negativo: 8


**Pregunta 3:** ¿Por qué un registro *huérfano* es un problema en una base relacional? Da un ejemplo con Farmacia MediSur.

**Respuesta:** _Porque comprometen la integridad de la base de datos, ocupan espacio sin aportar utilidad y afectan la calidad de las consultas. Los ejemplos con la farmacia MediSur son las ventas que no tienen un cliente asociado o un empleado asociado; si no podemos saber qué empleado hizo una venta ni a quién, ese dato no sirve para el análisis._

### Paso 6: TRANSFORM — limpiar la tabla clientes con una función
Ponemos **todas** las limpiezas dentro de una función reutilizable. Así el mismo pipeline servirá para cualquier fuente nueva (Paso 11).

In [ ]:
# Paso 6: TRANSFORM - limpieza de clientes en una funcion reutilizable
def transformar_clientes(df):
    df = df.copy()

    # 1. Eliminar documentos (DNI) duplicados, conservando el primero
    df = df.drop_duplicates(subset=["num_documento"], keep="first")

    # 2. Estandarizar texto (espacios y may/min)
    for col in ["nombres", "apellidos", "distrito", "segmento", "tipo_documento"]:
        df[col] = df[col].astype("string").str.strip().str.title()

    # 3. Corregir correos invalidos: los que no tienen formato valido se anulan
    formato_ok = df["correo"].str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", na=False)
    df["correo_fue_corregido"] = df["correo"].notna() & ~formato_ok
    df.loc[~formato_ok, "correo"] = pd.NA

    # 4. Completar valores faltantes con marcas claras
    df["correo"] = df["correo"].fillna("SIN CORREO REGISTRADO")
    df["telefono"] = df["telefono"].fillna("000000000")
    df["distrito"] = df["distrito"].fillna("No Especificado")

    # 5. Validar la fecha de registro y crear indicador de calidad
    df["fecha_registro_dt"] = pd.to_datetime(df["fecha_registro"], errors="coerce",
                                             format="mixed", dayfirst=True)
    df["fecha_valida"] = df["fecha_registro_dt"].notna()

    # 6. condicion_cronica es un DATO SENSIBLE (salud): se conserva, se gobierna en la S14
    return df

df_clientes_limpio = transformar_clientes(df_clientes)
print("Clientes antes :", len(df_clientes))
print("Clientes despues:", len(df_clientes_limpio), "(se quitaron DNI duplicados)")
display(df_clientes_limpio.head())

Clientes antes : 208
Clientes despues: 202 (se quitaron DNI duplicados)


,id_cliente,codigo_cliente,tipo_documento,num_documento,nombres,apellidos,correo,telefono,distrito,segmento,fecha_registro,condicion_cronica,correo_fue_corregido,fecha_registro_dt,fecha_valida
0,1,C0001,Ruc,79941052,Iván,Aguirre Bautista,iván1@correo.com,923380330,Rímac,Corporativo,2026-01-24,Hipertensión,False,2026-01-24,True
1,2,C0002,Dni,377593,Melissa,Torres Lévano,melissa2@correo.com,966971049,Villa El Salvador,Corporativo,2026-05-18,Asma,False,2026-05-18,True
2,3,C0003,Dni,38119160,Milagros,Villanueva Zevallos,milagros3@correo.com,911285064,Surco,Preferente,2026-03-08,Ninguna,False,2026-03-08,True
3,4,C0004,Dni,14282620,Karla,Ríos Ninaquispe,karla4@correo.com,999382396,Villa El Salvador,Regular,2026-05-10,Ninguna,False,2026-05-10,True
4,5,C0005,Dni,12191834,Hugo,Romero Castillo,hugo5@correo.com,982000067,Surco,Corporativo,2026-04-20,Asma,False,2026-04-20,True


**Pregunta 4:** ¿Qué transformaciones aplicó la función y por qué son necesarias? ¿Por qué `condicion_cronica` no se elimina?

**Respuesta:** _Se eliminaron números de DNI duplicados, se estandarizó el texto, se anularon los correos con formato inválido, se completaron valores faltantes y se validó la fecha de registro. La variable condicion_crónica se conserva hasta el desarrollo de la semana 14._

### Paso 7: TRANSFORM — corregir la tabla operaciones (ventas)
Corregimos los montos negativos (a cero, marcando el ajuste) y marcamos las ventas huérfanas para no perder trazabilidad.

In [ ]:
# Paso 7: TRANSFORM - corregir operaciones
df_op = pd.read_sql_query("SELECT * FROM operaciones", conn_origen)
ids_cliente = set(pd.read_sql_query("SELECT id_cliente FROM clientes", conn_origen)["id_cliente"])
ids_empleado = set(pd.read_sql_query("SELECT id_empleado FROM empleados", conn_origen)["id_empleado"])

def transformar_operaciones(df):
    df = df.copy()
    df["monto_original"] = df["monto_total"]
    df["monto_ajustado"] = np.where(df["monto_total"] < 0, 0, df["monto_total"])
    df["monto_fue_ajustado"] = df["monto_total"] < 0
    df["cliente_valido"] = df["id_cliente"].isin(ids_cliente)
    df["empleado_valido"] = df["id_empleado"].isin(ids_empleado)
    return df

df_op_limpio = transformar_operaciones(df_op)
print("Ventas procesadas:", len(df_op_limpio))
print("Montos ajustados:", int(df_op_limpio["monto_fue_ajustado"].sum()))
print("Ventas marcadas sin cliente valido:", int((~df_op_limpio["cliente_valido"]).sum()))
print("Ventas marcadas sin empleado valido:", int((~df_op_limpio["empleado_valido"]).sum()))
display(df_op_limpio.head())

Ventas procesadas: 1010
Montos ajustados: 8
Ventas marcadas sin cliente valido: 15
Ventas marcadas sin empleado valido: 22


,id_operacion,codigo_operacion,id_cliente,id_sede,id_empleado,fecha_operacion,canal,estado,monto_total,monto_original,monto_ajustado,monto_fue_ajustado,cliente_valido,empleado_valido
0,1,O00001,69,4,11.0,2026-06-04,Presencial,Completado,3341.39,3341.39,3341.39,False,True,True
1,2,O00002,139,3,40.0,2026-05-01,Delivery,Observado,1218.32,1218.32,1218.32,False,True,True
2,3,O00003,79,3,26.0,2026-02-26,Presencial,Completado,3777.17,3777.17,3777.17,False,True,True
3,4,O00004,89,3,3.0,2026-03-27,Web,Observado,2541.42,2541.42,2541.42,False,True,True
4,5,O00005,79,6,6.0,2026-06-27,App,Cancelado,-1767.60,-1767.60,0.00,True,True,True


### Paso 8: resumen de transformaciones

In [ ]:
# Paso 8: resumen de las transformaciones aplicadas
correo = df_clientes["correo"].dropna()
correos_malos = int((~correo.str.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", na=False)).sum())

resumen_etl = pd.DataFrame([
    ["Clientes: DNI duplicados eliminados", int(df_clientes["num_documento"].duplicated().sum())],
    ["Clientes: correos nulos completados", int(df_clientes["correo"].isna().sum())],
    ["Clientes: telefonos nulos completados", int(df_clientes["telefono"].isna().sum())],
    ["Clientes: distritos nulos completados", int(df_clientes["distrito"].isna().sum())],
    ["Clientes: correos invalidos corregidos", correos_malos],
    ["Ventas: montos negativos ajustados", int(df_op_limpio["monto_fue_ajustado"].sum())],
    ["Ventas: huerfanas sin cliente", int((~df_op_limpio["cliente_valido"]).sum())],
    ["Ventas: huerfanas sin empleado", int((~df_op_limpio["empleado_valido"]).sum())],
], columns=["control", "cantidad"])

display(resumen_etl)

,control,cantidad
0,Clientes: DNI duplicados eliminados,6
1,Clientes: correos nulos completados,14
2,Clientes: telefonos nulos completados,18
3,Clientes: distritos nulos completados,10
4,Clientes: correos invalidos corregidos,11
5,Ventas: montos negativos ajustados,8
6,Ventas: huerfanas sin cliente,15
7,Ventas: huerfanas sin empleado,22


### Paso 9: LOAD — cargar los datos limpios a una nueva base
Guardamos las tablas limpias en `farmacia_migrada.db`. La base original **no se toca**: la migración siempre escribe en un destino nuevo.

In [ ]:
# Paso 9: LOAD - cargar a la base destino
conn_destino = sqlite3.connect(DB_DESTINO)

df_clientes_limpio.to_sql("clientes_limpio", conn_destino, if_exists="replace", index=False)
df_op_limpio.to_sql("ventas_limpia", conn_destino, if_exists="replace", index=False)
resumen_etl.to_sql("resumen_etl", conn_destino, if_exists="replace", index=False)
conn_destino.commit()

print("Datos cargados en la base destino.")
display(pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn_destino))

Datos cargados en la base destino.


,name
0,clientes_limpio
1,resumen_etl
2,ventas_limpia


**Pregunta 5:** ¿Por qué la migración escribe en una base **nueva** en lugar de modificar la original?

**Respuesta:** _La definición de migración implica mover datos de un lugar a otro, y es muy importante preservar los datos originales, de manera que si algo sale mal, siempre será posible volver a iniciar las operaciones desde cero._

### Paso 10: validación post-migración
Comparamos origen y destino: los clientes del destino deben ser los del origen menos los DNI duplicados.

In [ ]:
# Paso 10: validacion post-migracion
clientes_origen = len(df_clientes)
clientes_destino = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM clientes_limpio", conn_destino)["n"][0]
dup_eliminados = int(df_clientes["num_documento"].duplicated().sum())

print("Clientes en origen :", clientes_origen)
print("Clientes en destino:", clientes_destino)
print("DNI duplicados eliminados:", dup_eliminados)
print("Validacion de conteo:", clientes_origen - dup_eliminados == clientes_destino)

validacion = pd.DataFrame([
    ["clientes_origen", clientes_origen],
    ["clientes_destino", clientes_destino],
    ["dni_duplicados_eliminados", dup_eliminados],
    ["correos_corregidos_en_destino",
     int(pd.read_sql_query("SELECT correo_fue_corregido FROM clientes_limpio", conn_destino)["correo_fue_corregido"].sum())],
    ["ventas_monto_ajustado",
     int(pd.read_sql_query("SELECT monto_fue_ajustado FROM ventas_limpia", conn_destino)["monto_fue_ajustado"].sum())],
], columns=["indicador", "valor"])

display(validacion)

Clientes en origen : 208
Clientes en destino: 202
DNI duplicados eliminados: 6
Validacion de conteo: True


,indicador,valor
0,clientes_origen,208
1,clientes_destino,202
2,dni_duplicados_eliminados,6
3,correos_corregidos_en_destino,11
4,ventas_monto_ajustado,8


**Pregunta 6:** ¿La migración fue correcta? Justifica con los conteos de origen y destino.

**Respuesta:** _La migración fue correcta. Se comprueba que los clientes al origen son iguales a los de destino más los duplicados eliminados._

### Paso 11: segunda fuente — una sucursal aliada llega en CSV
La cadena sumó una **sucursal aliada** que exporta a sus clientes con **otros nombres de columna** (`DOC`, `NOMBRE`, `APELLIDO`, `EMAIL`, `CEL`, `ZONA`). Antes de unificarla al destino, hay que renombrar sus columnas al esquema estándar.

In [ ]:
# Paso 11: generar el CSV de la sucursal aliada (fuente nueva con columnas distintas)
np.random.seed(11)
nombres = ["Ana Torres", "Luis Ramirez", "Carmen Vega", "Marco Huaman",
           "Rosa Castillo", "Diego Flores", "Julia Paredes", "Pedro Quispe"]
filas_ext = []
for i in range(1, 26):
    nom = np.random.choice(nombres).split()
    correo = f"{nom[0].lower()}@correo.com" if np.random.rand() > 0.1 else "correo_malo"
    filas_ext.append([9000 + i, f"C9{i:04d}", "DNI", f"7{i:07d}",
                      nom[0], nom[1], correo,
                      f"9{np.random.randint(10000000, 99999999)}",
                      np.random.choice(["Surco", "  Ate ", "lince", None]),
                      "Nuevo", "2026-05-15", "Ninguna"])

df_ext = pd.DataFrame(filas_ext, columns=[
    "id_cliente", "codigo_cliente", "tipo_documento", "DOC", "NOMBRE", "APELLIDO",
    "EMAIL", "CEL", "ZONA", "segmento", "fecha_registro", "condicion_cronica"])
df_ext.to_csv("clientes_sucursal_aliada.csv", index=False)

df_nueva = pd.read_csv("clientes_sucursal_aliada.csv")
mapa = {"DOC": "num_documento", "NOMBRE": "nombres", "APELLIDO": "apellidos",
        "EMAIL": "correo", "CEL": "telefono", "ZONA": "distrito"}
df_nueva = df_nueva.rename(columns=mapa)
print("Columnas unificadas:", list(df_nueva.columns))
display(df_nueva.head())

Columnas unificadas: ['id_cliente', 'codigo_cliente', 'tipo_documento', 'num_documento', 'nombres', 'apellidos', 'correo', 'telefono', 'distrito', 'segmento', 'fecha_registro', 'condicion_cronica']


,id_cliente,codigo_cliente,tipo_documento,num_documento,nombres,apellidos,correo,telefono,distrito,segmento,fecha_registro,condicion_cronica
0,9001,C90001,DNI,70000001,Luis,Ramirez,correo_malo,946379739,Ate,Nuevo,2026-05-15,Ninguna
1,9002,C90002,DNI,70000002,Pedro,Quispe,pedro@correo.com,969930273,NaN,Nuevo,2026-05-15,Ninguna
2,9003,C90003,DNI,70000003,Carmen,Vega,carmen@correo.com,989979184,Ate,Nuevo,2026-05-15,Ninguna
3,9004,C90004,DNI,70000004,Ana,Torres,correo_malo,973171197,Surco,Nuevo,2026-05-15,Ninguna
4,9005,C90005,DNI,70000005,Carmen,Vega,carmen@correo.com,991192778,Ate,Nuevo,2026-05-15,Ninguna


**Pregunta 7:** ¿Qué pasaría si cargáramos la fuente nueva **sin** renombrar sus columnas? ¿Por qué la migración exige un esquema común?

**Respuesta:** _Si cargamos a la base de datos valores dentro de una columna con un nombre distinto al estándar, la base de datos no tendría forma de reconocer los valores como parte de la información que usualmente almacena, por lo que almacenaría una nueva tabla como un ente aislado del resto, sin relación ni capacidad de aportar utilidad, además de ocupar espacio necesario. El esquema común asegura que los datos de un atributo tengan el mismo formato, guarden una relación con las otras entidades y puedan ser ingresados al mismo pipeline reutilizable._

### Paso 12: función reutilizable — un stored procedure simulado
En SQL Server o PostgreSQL esto sería un **stored procedure** (`CREATE PROCEDURE`). En SQLite lo simulamos con una función de Python que ejecuta el pipeline completo cada vez que llega una fuente nueva.

In [ ]:
# Paso 12: funcion reutilizable - stored procedure simulado
def cargar_clientes(df_fuente, conn, tabla="clientes_limpio"):
    """Limpia una fuente con el pipeline del Paso 6 y la agrega a la tabla destino."""
    df = transformar_clientes(df_fuente)
    df.to_sql(tabla, conn, if_exists="append", index=False)
    return len(df)

n_aliada = cargar_clientes(df_nueva, conn_destino)
print("Clientes de la sucursal aliada cargados:", n_aliada)

Clientes de la sucursal aliada cargados: 25


**Pregunta 8:** ¿Qué ventaja tiene envolver el pipeline en una función reutilizable? ¿Qué relación tiene con un stored procedure?

**Respuesta:** _Un pipeline dentro de una función puede ser reutilizado de forma indefinida con cada conjunto de datos relevante para dicho pipeline, siendo posible incluso adaptarlo para otros trabajos con pocos ajustes. En SQLite, es el equivalente o la emulación de un procedimiento almacenado en una base de datos no embebida._

### Paso 13: checklist de validación de la Semana 10

In [ ]:
# Paso 13: checklist de validacion
destino = pd.read_sql_query("SELECT * FROM clientes_limpio", conn_destino)
total_destino = len(destino)
esperado = (clientes_origen - dup_eliminados) + n_aliada

checklist = pd.DataFrame([
    ["Registros esperados = registros en destino", bool(esperado == total_destino)],
    ["Cero nulos en columnas clave (nombres, correo)", bool(destino[["nombres", "correo"]].isna().sum().sum() == 0)],
    ["Sin DNI duplicados dentro de la base migrada original", bool(df_clientes_limpio["num_documento"].duplicated().sum() == 0)],
    ["Correo nunca queda vacio (usa marca)", bool((destino["correo"].str.strip() != "").all())],
    ["Distrito nunca queda nulo", bool(destino["distrito"].isna().sum() == 0)],
], columns=["verificacion", "cumple"])

display(checklist)

,verificacion,cumple
0,Registros esperados = registros en destino,True
1,"Cero nulos en columnas clave (nombres, correo)",True
2,Sin DNI duplicados dentro de la base migrada o...,True
3,Correo nunca queda vacio (usa marca),True
4,Distrito nunca queda nulo,True


**Pregunta 9 (reto):** El pipeline dejó pasar un problema: al **unir** la base original con la sucursal aliada podría haber **DNI repetidos entre ambas fuentes**. Detéctalo con `destino["num_documento"].value_counts()` y explica cómo lo corregirías.

**Respuesta:** _De presentarse ese problema, o como prevención, puede agregarse un bloque de código justo después del stored procedure, porque solamente entonces puede aparecer este problema, el bloque de código añadido se encargaría de leer la tabla completa con los nuevos datos, luego usaría drop_duplicates y el resultado de la operación anterior lo usaría para sobreescribir la tabla destino. En bases de datos no embebidas, este problema se arreglaría directamente en el servidor de la base de datos._

In [ ]:
#Bloque de código para la pregunta 9
dnis_duplicados = destino["num_documento"].value_counts()
registros_duplicados = dnis_duplicados[dnis_duplicados > 1].index
print("DNI Duplicados:", registros_duplicados)
total_duplicados = dnis_duplicados[dnis_duplicados > 1].sum()
print(f"\nNúmero total de registros duplicados: {total_duplicados}")

DNI Duplicados: Index([], dtype='object', name='num_documento')

Número total de registros duplicados: 0


In [ ]:
#Bloque adicional para responder la pregunta C de la actividad 3
clientes_dimensiones = df_clientes.shape
clientes_limpio_dimensiones = df_clientes_limpio.shape
print("Dimensiones de clientes:", clientes_dimensiones)
print("Dimensiones de clientes_limpio:", clientes_limpio_dimensiones)

Dimensiones de clientes: (208, 12)
Dimensiones de clientes_limpio: (202, 15)


### Paso 14: reporte para el negocio y exportación

In [ ]:
# Paso 14: reporte para el negocio + exportar + cerrar
reporte = pd.read_sql_query("""
    SELECT distrito, segmento, COUNT(*) AS total_clientes
    FROM clientes_limpio
    GROUP BY distrito, segmento
    ORDER BY total_clientes DESC""", conn_destino)
display(reporte.head(10))

destino.to_csv("clientes_limpios_farmacia.csv", index=False, encoding="utf-8-sig")
resumen_etl.to_csv("resumen_etl_farmacia.csv", index=False, encoding="utf-8-sig")
validacion.to_csv("validacion_etl_farmacia.csv", index=False, encoding="utf-8-sig")
checklist.to_csv("checklist_etl_farmacia.csv", index=False, encoding="utf-8-sig")

print("Archivos generados:")
for a in ["clientes_limpios_farmacia.csv", "resumen_etl_farmacia.csv",
          "validacion_etl_farmacia.csv", "checklist_etl_farmacia.csv",
          "farmacia_migrada.db"]:
    print("   -", a)

conn_origen.close()
conn_destino.close()
print("\nLaboratorio finalizado correctamente.")

,distrito,segmento,total_clientes
0,No Especificado,Nuevo,12
1,Lince,Nuevo,10
2,Surco,Nuevo,8
3,Ate,Nuevo,7
4,Rímac,Regular,7
5,Chorrillos,Corporativo,6
6,San Juan De Lurigancho,Corporativo,6
7,Ate,Regular,5
8,Breña,Regular,5
9,Los Olivos,Preferente,5


Archivos generados:
   - clientes_limpios_farmacia.csv
   - resumen_etl_farmacia.csv
   - validacion_etl_farmacia.csv
   - checklist_etl_farmacia.csv
   - farmacia_migrada.db

Laboratorio finalizado correctamente.


## Actividad 3: caso de estudio — cierre de la migración MediSur

**Pregunta A:** ¿Qué problemas de calidad e integridad traía la base oficial de Farmacia MediSur y cuáles corrigió tu pipeline?

**Respuesta:** _La base oficial de MediSur venía con números de DNI duplicados, texto sin estandarizar, correos con formato inválido, valores faltantes y fechas de registro sin validar. El pipeline corrigió todos los problemas mencionados._

**Pregunta B:** ¿Qué controles de validación aplicaste y qué resultados obtuviste? Compara los conteos de origen y destino.

**Respuesta:** _Se hizo un conteo del número de filas de la tabla clientes antes y después de la migración, obteniendo 208 filas al inicio y 202 filas al final. Se comprobó luego que el número de registros inicial es igual al número de registros después de la migración más el número de duplicados de DNI eliminados._

**Pregunta C:** Muestra el antes y después con `df.shape`. ¿Hubo pérdida intencional de datos (duplicados)? Justifica.

**Respuesta:** _Usando la función shape, ejecutada antes de cerrar la conexión, se puede ver que la tabla clientes tiene 208 filas, y la tabla clientes_limpio tiene 202 filas, lo que indica que hubo pérdida intencional de datos duplicados, puesto que su inclusión alteraría los cálculos y métricas que se pretenden realizar._

## Actividad final

Redacta tres conclusiones breves:

1. ¿Qué aprendiste sobre ETL y migración de datos?

La migración de datos, que incluye al ETL, es un proceso complejo, que necesita planificación, un flujo de datos reproducible, automatizado y documentado, y también una validación de completitud e integridad.

2. ¿Qué problema de calidad o de integridad te pareció el más importante?

Me llamó la atención un número de DNI, que no es nulo, pero solo tiene 6 cifras. No es algo muy importante, pues se puede corregir en la próxima compra del cliente, mas el código, tal como está, no tiene forma de detectarlo, completarlo o corregirlo. El problema más importante, según mi parecer, lo representan las ventas huérfanas (integridad), que si bien pueden indicar un error humano, también pueden levantar sospechas de actividades ilícitas.

3. Esta base limpia es la base de tu proyecto. ¿Cómo te servirá para las semanas siguientes (metadatos, calidad, gobierno)?

Las consultas que se hicieron en este laboratorio pueden servir para abordar la parte técnica de los metadatos, los diagnósticos y validaciones servirán para mejorar la calidad de los datos, y la documentación de los cambios será útil para la parte de la gobernanza concerniente a trazabilidad y auditoría.